# 03 — Mô hình 3: PhoBERT (Transformer-based)

Fine-tune `vinai/phobert-base-v2` dạng **cross-encoder**: premise và hypothesis đi vào
cùng một chuỗi `<s> premise </s></s> hypothesis </s>`, nên self-attention nhìn thấy
cả hai câu cùng lúc. Đây là khác biệt then chốt so với TextCNN/BiLSTM ở notebook 02
(hai câu được mã hóa độc lập rồi mới ghép) — và là lý do Transformer thường bỏ xa
hai mô hình kia trên NLI.

**Bắt buộc:** PhoBERT được huấn luyện trên văn bản đã tách từ, nên phải chạy
`underthesea.word_tokenize` trước tokenizer. Bỏ bước này thường mất vài điểm accuracy —
notebook có nhánh ablation để đo chính xác con số đó.

Yêu cầu: GPU (Settings → Accelerator → GPU T4 x2 hoặc P100). Chạy `01_eda.ipynb` trước.

## Setup Kaggle — kéo code từ GitHub

Chạy cell này **trước tiên** trên Kaggle. Bỏ qua được khi chạy ở máy local.
Sửa code ở máy → `git push` → chạy lại cell này để lấy bản mới.

In [ ]:
import os

REPO_URL = "https://github.com/dofu18/ViANLI_DL_NLP.git"

if os.path.exists("/kaggle/input"):
    !rm -rf /kaggle/working/repo
    !git clone -q $REPO_URL /kaggle/working/repo
    !cp -r /kaggle/working/repo/src /kaggle/working/
    !cp -r /kaggle/working/repo/configs /kaggle/working/
    !cp -r /kaggle/working/repo/data /kaggle/working/     # split cố định từ 01_eda
    print("src/:", sorted(os.listdir("/kaggle/working/src")))
else:
    print("Chạy local — bỏ qua bước clone.")

In [ ]:
# BẮT BUỘC: Kaggle không cài sẵn underthesea. Thiếu nó thì word_segment() raise
# và nhánh PhoBERT dừng ngay, thay vì âm thầm train trên text chưa tách từ.
!pip install -q underthesea "transformers>=4.44" "datasets>=2.21" accelerate

import json, os, sys, time

import numpy as np
import pandas as pd
import torch

ON_KAGGLE = os.path.exists("/kaggle/input")
ROOT = "/kaggle/working" if ON_KAGGLE else os.path.abspath("..")
sys.path.insert(0, os.path.join(ROOT, "src"))

FIG_DIR = os.path.join(ROOT, "outputs", "figures")
LOG_DIR = os.path.join(ROOT, "outputs", "logs")
CKPT_DIR = os.path.join(ROOT, "outputs", "checkpoints")
for d in (FIG_DIR, LOG_DIR, CKPT_DIR):
    os.makedirs(d, exist_ok=True)

from data import LABELS, label_to_id, load_splits, normalize, set_seed, word_segment
from models import build_transformer, count_params
from evaluate import (error_examples, full_report, hf_compute_metrics,
                      plot_confusion_matrix, plot_learning_curve)

SEED = 42
set_seed(SEED)
assert torch.cuda.is_available(), "Bật GPU trong Settings của Kaggle Notebook"
print(torch.cuda.get_device_name(0))

## 1. Nạp split cố định

Cùng nguồn split với notebook 02 — đây là điều kiện của "nguyên tắc so sánh công bằng".

In [ ]:
from datasets import Dataset, DatasetDict

SPLIT_DIR = os.path.join(ROOT, "data", "splits")
COLS = {"premise": "premise", "hypothesis": "hypothesis", "label": "label"}

if os.path.exists(os.path.join(SPLIT_DIR, "train.csv")):
    ds = DatasetDict({
        name: Dataset.from_pandas(
            pd.read_csv(os.path.join(SPLIT_DIR, f"{name}.csv"), encoding="utf-8")
              .fillna({"premise": "", "hypothesis": ""}))
        for name in ("train", "validation", "test")
    })
    cols = COLS
    print("Nạp từ data/splits/ (split cố định)")
else:
    ds, cols = load_splits(seed=SEED)
    print("CẢNH BÁO: chưa có data/splits/ — chạy 01_eda.ipynb trước")

{k: len(v) for k, v in ds.items()}

## 2. Tokenize

`word_segment` tốn thời gian nên cache lại kết quả một lần, dùng chung cho cả nhánh
chính lẫn nhánh ablation.

In [ ]:
MODEL_ID = "vinai/phobert-base-v2"
REVISION = "86cd7fd4c148980922ac11a2cf5e257f2ba639e1"   # pin commit hash (yêu cầu của đề)
MAX_LENGTH = 128     # chốt theo p95 của 01_eda (premise 51 + hypothesis 28 từ ~ 120 subword)

t0 = time.time()
seg_cache = {}

def prep(text, segment=True):
    text = normalize(text)
    if not segment:
        return text
    if text not in seg_cache:
        seg_cache[text] = word_segment(text)
    return seg_cache[text]

# Fail fast: PhoBERT-base-v2 giả định đầu vào ĐÃ tách từ. Bản chạy trước rơi vào
# fallback im lặng nên ablation no_wordseg trùng khít baseline (Δ = 0.000, vô nghĩa).
_probe = prep("Tọa đàm được tổ chức tại Hà Nội")
assert "_" in _probe, f"Tách từ KHÔNG hoạt động ({_probe!r}). Cài underthesea rồi restart kernel."
print("Tách từ OK:", _probe)

print("Ví dụ tách từ:")
print(" ", prep(ds["train"][cols["premise"]][0]))
print(" ", prep(ds["train"][cols["hypothesis"]][0]))

In [ ]:
tokenizer, _ = build_transformer(MODEL_ID, revision=REVISION)

def make_encoded(segment=True, hypothesis_only=False, max_length=MAX_LENGTH):
    def tok(batch):
        n = len(batch[cols["label"]])
        premises = [""] * n if hypothesis_only else \
            [prep(x, segment) for x in batch[cols["premise"]]]
        out = tokenizer(premises,
                        [prep(x, segment) for x in batch[cols["hypothesis"]]],
                        truncation=True, max_length=max_length)
        out["labels"] = [label_to_id(y) for y in batch[cols["label"]]]
        return out
    return ds.map(tok, batched=True, remove_columns=ds["train"].column_names)

encoded = make_encoded()
print(f"Tokenize xong sau {time.time() - t0:.1f}s")

# Kiểm tra tỉ lệ bị cắt cụt — nếu cao thì phải tăng MAX_LENGTH
lens = [len(x) for x in encoded["train"]["input_ids"]]
print(f"độ dài subword: p50={np.percentile(lens, 50):.0f} "
      f"p95={np.percentile(lens, 95):.0f} max={max(lens)}")
print(f"tỉ lệ chạm trần {MAX_LENGTH}: {np.mean(np.array(lens) >= MAX_LENGTH):.2%}")
print(tokenizer.decode(encoded["train"][0]["input_ids"]))

## 3. Hàm chạy một thí nghiệm

Cùng giao thức với notebook 02: chọn checkpoint tốt nhất theo **macro-F1 trên dev**,
test set chỉ chạy một lần ở cuối.

In [ ]:
from transformers import (DataCollatorWithPadding, EarlyStoppingCallback,
                          Trainer, TrainingArguments)

BASE = dict(learning_rate=2e-5, batch_size=32, grad_accum=1, epochs=8,
            warmup_ratio=0.06, weight_decay=0.01, patience=3)

RESULTS = {}

def run(run_name, enc, model_id=MODEL_ID, **overrides):
    cfg = {**BASE, **overrides}
    set_seed(SEED)
    _, model = build_transformer(model_id, revision=REVISION)

    args = TrainingArguments(
        output_dir=os.path.join(CKPT_DIR, run_name),
        learning_rate=cfg["learning_rate"],
        per_device_train_batch_size=cfg["batch_size"],
        per_device_eval_batch_size=cfg["batch_size"] * 2,
        gradient_accumulation_steps=cfg["grad_accum"],
        num_train_epochs=cfg["epochs"],
        warmup_ratio=cfg["warmup_ratio"],
        weight_decay=cfg["weight_decay"],
        fp16=True,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        logging_strategy="epoch",
        logging_dir=os.path.join(LOG_DIR, run_name),
        seed=SEED,
        report_to=[],
    )
    trainer = Trainer(
        model=model, args=args,
        train_dataset=enc["train"], eval_dataset=enc["validation"],
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=hf_compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=cfg["patience"])],
    )

    print(f"\n=== {run_name} | {count_params(model):,} tham số ===")
    torch.cuda.reset_peak_memory_stats()
    started = time.time()
    trainer.train()
    elapsed = time.time() - started

    logs = trainer.state.log_history
    losses = {int(r["epoch"]): r["loss"] for r in logs if "loss" in r}
    history = [{"epoch": int(r["epoch"]),
                "train_loss": losses.get(int(r["epoch"]), float("nan")),
                "val_macro_f1": r["eval_macro_f1"],
                "val_accuracy": r["eval_accuracy"]}
               for r in logs if "eval_macro_f1" in r]

    # history dạng dữ liệu (không chỉ dạng hình) — nb 05 và báo cáo cần file này
    with open(os.path.join(LOG_DIR, f"{run_name}_history.json"), "w",
              encoding="utf-8") as f:
        json.dump(history, f, ensure_ascii=False, indent=2)

    pred = trainer.predict(enc["test"])
    y_true, y_pred = pred.label_ids, pred.predictions.argmax(-1)
    report = full_report(y_true, y_pred,
                         out_json=os.path.join(LOG_DIR, f"{run_name}_test.json"))
    plot_learning_curve(history, run_name,
                        os.path.join(FIG_DIR, f"{run_name}_curve.png"))
    plot_confusion_matrix(y_true, y_pred, run_name,
                          os.path.join(FIG_DIR, f"{run_name}_cm.png"))

    # thời gian suy luận cho §Chi phí tính toán
    t1 = time.time(); trainer.predict(enc["test"])
    ms_per_sample = (time.time() - t1) / len(enc["test"]) * 1000

    summary = {"run_name": run_name, "model_id": model_id,
               "accuracy": report["accuracy"], "macro_f1": report["macro_f1"],
               "weighted_f1": report["weighted_f1"],
               "params": count_params(model),
               "epochs_chạy": len(history),
               "train_seconds": round(elapsed, 1),
               "inference_ms_per_sample": round(ms_per_sample, 2),
               "peak_vram_gb": round(torch.cuda.max_memory_allocated() / 1024 ** 3, 2),
               "word_segment": enc is encoded}
    with open(os.path.join(LOG_DIR, f"{run_name}_summary.json"), "w",
              encoding="utf-8") as f:
        json.dump({**summary, "config": cfg}, f, ensure_ascii=False, indent=2)
    RESULTS[run_name] = {**summary, "_preds": (y_true, y_pred),
                         "_history": history}
    if run_name == "phobert_base_v2":
        best_dir = os.path.join(CKPT_DIR, run_name, "best")
        trainer.save_model(best_dir)
        tokenizer.save_pretrained(best_dir)
        print("Đã lưu:", best_dir)
    # không giữ trainer trong RESULTS: 4 bản PhoBERT cùng lúc là thừa VRAM/RAM
    del trainer, model
    torch.cuda.empty_cache()
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    return summary

## 4. Fine-tune PhoBERT

Khoảng 8 epoch, early stopping patience 3. Trên T4 với batch 32 và `max_length=256`,
mỗi epoch thường mất vài phút — canh quota GPU.

In [ ]:
run("phobert_base_v2", encoded)

In [ ]:
# (Việc lưu model tốt nhất đã chuyển vào trong run() để giải phóng VRAM ngay
#  sau mỗi lần chạy — xem cell định nghĩa run() ở trên.)
print("best checkpoint:", os.path.join(CKPT_DIR, "phobert_base_v2", "best"))

## 5. Ablation

| Nhánh | Câu hỏi |
|---|---|
| `no_wordseg` | Bỏ tách từ mất bao nhiêu điểm? (PhoBERT pretrain trên văn bản đã tách từ) |
| `hyponly` | PhoBERT đoán được nhãn khi không thấy premise? So với BiLSTM ở notebook 02 |
| `lr_5e-5` | Learning rate ảnh hưởng thế nào? |

Mỗi nhánh tốn thêm một lần train — bỏ bớt nếu hết quota, nhưng phải ghi rõ trong báo cáo.

In [ ]:
run("phobert_no_wordseg", make_encoded(segment=False))

In [ ]:
run("phobert_hyponly", make_encoded(hypothesis_only=True))

In [ ]:
run("phobert_lr5e-5", encoded, learning_rate=5e-5)

## 6. Tổng hợp

In [ ]:
table = pd.DataFrame([
    {k: v for k, v in r.items() if not k.startswith("_")}
    for r in RESULTS.values()
]).sort_values("accuracy", ascending=False)
display(table.round(4))
table.to_csv(os.path.join(LOG_DIR, "phobert_results.csv"), index=False)

# So với kết quả CNN/RNN nếu notebook 02 đã chạy
prev = os.path.join(LOG_DIR, "cnn_rnn_results.csv")
if os.path.exists(prev):
    display(pd.concat([pd.read_csv(prev), table], ignore_index=True)
              .sort_values("accuracy", ascending=False)
              [["run_name", "accuracy", "macro_f1", "params", "train_seconds"]]
              .round(4))

In [ ]:
# Lưu dự đoán trên test để 05_analysis.ipynb phân tích lỗi chéo giữa các mô hình
pred_dir = os.path.join(LOG_DIR, "predictions")
os.makedirs(pred_dir, exist_ok=True)
for name, r in RESULTS.items():
    np.save(os.path.join(pred_dir, f"{name}.npy"), r["_preds"][1])
y_true_path = os.path.join(pred_dir, "y_true.npy")
y_true_new = np.asarray(RESULTS[list(RESULTS)[0]]["_preds"][0])
if os.path.exists(y_true_path):
    assert np.array_equal(np.load(y_true_path), y_true_new), (
        "y_true lệch so với file đã lưu — thứ tự test set không nhất quán giữa các "
        "mô hình, mọi so sánh chéo ở nb 05 sẽ sai.")
else:
    np.save(y_true_path, y_true_new)
print(sorted(os.listdir(pred_dir)))

In [ ]:
best = table.iloc[0]["run_name"]
y_true, y_pred = RESULTS[best]["_preds"]
print(f"Tốt nhất: {best}")
print("Phân bố dự đoán:",
      {LABELS[i]: int((y_pred == i).sum()) for i in range(len(LABELS))})

pd.set_option("display.max_colwidth", 100)
pd.DataFrame(error_examples(ds["test"], cols, y_true, y_pred, n=10))

## Ghi chú cho Kaggle

- Dùng **Save Version → Save & Run All** để chạy nền; session tương tác 12h dễ ngắt giữa chừng.
- `save_total_limit=1` để `/kaggle/working` không vượt giới hạn 20GB.
- Nếu OOM: hạ `batch_size` xuống 16 và đặt `grad_accum=2` (effective batch giữ nguyên 32).
- Nếu notebook không có Internet: upload PhoBERT thành Kaggle Dataset rồi đặt
  `MODEL_ID = "/kaggle/input/phobert-base-v2"`.

## Cần điền vào báo cáo

- Bảng cấu hình mô hình 3: model ID, revision, số layer/head/hidden, số tham số, chiến lược fine-tune.
- Bảng siêu tham số (cột Transformer).
- Hình `phobert_base_v2_curve.png`, `phobert_base_v2_cm.png`.
- Bảng ablation từ `phobert_results.csv` — đặc biệt con số **mất bao nhiêu điểm khi bỏ tách từ**.
- Chênh lệch PhoBERT so với TextCNN/BiLSTM → luận điểm về vai trò của cross-attention.